In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from pathlib import Path


In [ ]:
# 0) 필요한 패키지(최초 1회만)
# pip install geopandas shapely pyproj matplotlib contextily

import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from pathlib import Path

# 1) 파일 경로 지정 (여기만 바꾸세요)
shp_path = Path('./LSMD_ADM_SECT_UMD_대전/LSMD_ADM_SECT_UMD_30_202510.shp')  # 예시(윈도우)
# shp_path = Path("/Users/you/Downloads/읍면동_경계.shp")  # 예시(mac)

gdf = gpd.read_file(shp_path, encoding='euc-kr')  # 한글 인코딩 주의
dunsan = gdf[gdf['EMD_NM'] == '둔산동']

dunsan_4326 = dunsan.to_crs(epsg=4326)
dunsan_4326

### 경계 표시

In [ ]:
import folium

b = dunsan_4326.total_bounds  # [minx, miny, maxx, maxy]
center = [(b[1]+b[3])/2, (b[0]+b[2])/2]

m = folium.Map(location=center, zoom_start=13, tiles="CartoDB positron")
folium.GeoJson(
    dunsan_4326.to_json(),
    name="둔산동 경계",
    style_function=lambda f: {"color":"#1f77b4","weight":2,"fill":False},
).add_to(m)
m.fit_bounds([[b[1], b[0]], [b[3], b[2]]])
m

## 정류장 마커

In [ ]:
import pandas as pd

bus_stop = pd.read_csv('./data/TalkFile_대전광역시_시내버스 기반정보_20250424 (1).CSV.csv', encoding='euc-kr')
bus_stop

gdf_stops = gpd.GeoDataFrame(
    bus_stop,
    geometry=gpd.points_from_xy(bus_stop["엑스좌표"], bus_stop["와이좌표"]),
    crs="EPSG:4326"
)
gdf_stops.head(2)

for _, row in gdf_stops.iterrows():
    lat = row["와이좌표"]
    lon = row["엑스좌표"]
    name = row["정류장명"]
    
    folium.CircleMarker(
        location=[lat, lon],
        radius=4,
        color="red",
        fill=True,
        fill_opacity=0.7,
        tooltip=name  # 마우스 올리면 정류장명 표시
    ).add_to(m)

m

## 편의점

In [ ]:
cv_df = pd.read_csv('./data/안전상비의약품_둔산동_WGS84.csv', dtype=str)
cv_df = cv_df[~cv_df['lat'].isnull()]

In [ ]:
gdf_convenience_stores = gpd.GeoDataFrame(
    cv_df,
    geometry=gpd.points_from_xy(cv_df["lon"], cv_df["lat"]),
    crs="EPSG:4326"
)
gdf_convenience_stores.head(2)

for _, row in gdf_convenience_stores.iterrows():
    lat = row["lat"]
    lon = row["lon"]
    name = row["사업장명"]

    folium.CircleMarker(
        location=[lat, lon],
        radius=4,
        color="blue",
        fill=True,
        fill_opacity=0.7,
        tooltip=name  # 마우스 올리면 상호명 표시
    ).add_to(m)

m

In [ ]:
for _, r in gdf_convenience_stores.iterrows():
    folium.Circle(
        location=[r["lat"], r["lon"]],
        radius=150,                # 200m
        color="green",             # 테두리 초록
        weight=1,
        fill=True,
        fill_color="green",        # 채움도 초록
        fill_opacity=0.15
    ).add_to(m)

from folium import Element

legend_html = """
<div style="
    position: fixed; 
    bottom: 20px; left: 20px; 
    z-index: 9999; 
    background: white; 
    padding: 8px 10px; 
    border: 1px solid #bbb; 
    border-radius: 6px; 
    box-shadow: 0 1px 4px rgba(0,0,0,.3);
    font-size: 13px;
">
  <b>범례</b><br>
  <span style="display:inline-block;width:10px;height:10px;background:blue;border-radius:50%;margin-right:6px;"></span>
  편의점(파랑)<br>
  <span style="display:inline-block;width:10px;height:10px;background:red;border-radius:50%;margin-right:6px;"></span>
  버스정류장(빨강)
</div>
"""

m.get_root().html.add_child(Element(legend_html))
m

## 기존 수거함 시각화

In [ ]:
import folium

b = dunsan_4326.total_bounds  # [minx, miny, maxx, maxy]
center = [(b[1]+b[3])/2, (b[0]+b[2])/2]

m2 = folium.Map(location=center, zoom_start=13, tiles="CartoDB positron")
folium.GeoJson(
    dunsan_4326.to_json(),
    name="둔산동 경계",
    style_function=lambda f: {"color":"#1f77b4","weight":2,"fill":False},
).add_to(m2)
m2.fit_bounds([[b[1], b[0]], [b[3], b[2]]])


In [ ]:
df_origin = pd.read_excel('./data/TalkFile_대전광역시 서구_폐의약품 수거함 현황.xlsx.xlsx', engine='openpyxl')
df_origin_dunsan = df_origin[df_origin['법정동명'] == '둔산동']
df_origin_dunsan.reset_index(drop=True, inplace=True)

In [ ]:
gdf_origin = gpd.GeoDataFrame(
    df_origin_dunsan,
    geometry=gpd.points_from_xy(df_origin_dunsan["경도"], df_origin_dunsan["위도"]),
    crs="EPSG:4326"
)
gdf_origin.head(2)

for _, row in gdf_origin.iterrows():
    lat = row["위도"]
    lon = row["경도"]
    name = row["수거장소명"]

    folium.CircleMarker(
        location=[lat, lon],
        radius=4,
        color="blue",
        fill=True,
        fill_opacity=0.7,
        tooltip=name  # 마우스 올리면 상호명 표시
    ).add_to(m2)

m2

In [ ]:
stores_m = gdf_convenience_stores.to_crs(5179)
stops_m  = gdf_stops.to_crs(5179)

stores_buf = stores_m.copy()
stores_buf["geometry"] = stores_buf.geometry.buffer(200)  # 200m

joined = gpd.sjoin(stops_m, stores_buf[["geometry"]], predicate="within", how="left")
# joined에는 정류장마다 'index_right' = 들어간 편의점 인덱스가 생김

counts = joined.groupby("index_right").size().rename("bus_stops_200m")
stores_m = stores_m.join(counts, how="left")
stores_m["bus_stops_200m"] = stores_m["bus_stops_200m"].fillna(0).astype(int)

stores_out = stores_m.to_crs(4326).copy()
stores_out["lon"] = stores_out.geometry.x
stores_out["lat"] = stores_out.geometry.y

keep = ["bus_stops_150m", "lon", "lat"]
if "store_id" in stores_out.columns: keep = ["store_id"] + keep
elif "name" in stores_out.columns:   keep = ["name"] + keep

# 이름 컬럼: '사업장명'을 그대로 사용(필요하면 영문으로 통일)
stores_out["store_name"] = stores_out["사업장명"].astype(str)

# 최종 컬럼 선택: 이름을 맨 앞에
keep = ["store_name", "bus_stops_200m", "lon", "lat"]
stores_df = stores_out[keep].reset_index(drop=True)

# (선택) 많은 정류장 순으로 정렬
# stores_df = stores_df.sort_values("bus_stops_200m", ascending=False)

stores_df.to_csv('./data/편의점_버스정류장_둔산동_결과.csv', index=False)